In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import other libraries when needed
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load raw data
df = pd.read_parquet("..\dataDumper.parquet")

In [3]:
df[['phaseNb']] = df[['RatioPayload']].diff().abs().cumsum()

In [4]:
dt_s_tolerance = 0.5
original_dt_s = 5
df["missing_record"] = False
df.loc[(df["dt_s"] - original_dt_s).abs() > dt_s_tolerance, "missing_record"] = True
df["missing_record"] = df["missing_record"].shift(-1)

In [5]:
tire_nb = [6]
tire_columns_to_keep = [
    col
    for tire in tire_nb
    for col in [f"Pressure_Pa_{tire}", f"Temperature_K_{tire}", f"ColdPressure_Pa_{tire}"]
]
columns_to_keep = tire_columns_to_keep + ["missing_record", "phaseNb"]
filtered_df = df[columns_to_keep]

In [6]:
# Sélection des lignes avec au moins une valeur manquante
na_values_df = (
    filtered_df[filtered_df[tire_columns_to_keep].isna().any(axis=1)]
    .drop(columns=tire_columns_to_keep)
)

# Calcul de la différence d'index
index_series = na_values_df.index.to_series()
na_values_df["index_diff"] = index_series.shift(-1) - index_series

# Lignes correspondant à la fin d'une suite d'index consécutifs
end_of_sequence = na_values_df[na_values_df["index_diff"] > 1]
end_of_sequence

,missing_record,phaseNb,index_diff
9116,True,28.0,25301.0
34420,True,65.0,3920.0
38347,True,76.0,21207.0
59565,True,112.0,434.0
59999,True,114.0,5585.0
65592,True,132.0,21718.0
87323,True,204.0,1462.0
88809,False,212.0,28.0
88838,True,212.0,10136.0
98988,False,248.0,6140.0


In [7]:
~end_of_sequence["missing_record"]

9116      -2
34420     -2
38347     -2
59565     -2
59999     -2
65592     -2
87323     -2
88809     -1
88838     -2
98988     -1
105134    -1
109606    -2
115727    -1
133473    -2
157173    -2
166089    -2
170974    -2
Name: missing_record, dtype: object

In [8]:
print(f"Columns: {tire_columns_to_keep}")

print(
    f"Number of records with at least one missing value : "
    f"{len(end_of_sequence)}"
)

na_value_missing_position = len(end_of_sequence[end_of_sequence["missing_record"]])
print(
    f"Number of records with missing value that precedes missing record: "
    f"{na_value_missing_position}"
)

na_value_without_missing_position = len(end_of_sequence[~end_of_sequence["missing_record"] == -1])
print(
    f"Number of records with missing value that not precedes missing record: "
    f"{na_value_without_missing_position}"
)

Columns: ['Pressure_Pa_6', 'Temperature_K_6', 'ColdPressure_Pa_6']
Number of records with at least one missing value : 17
Number of records with missing value that precedes missing record: 13
Number of records with missing value that not precedes missing record: 4


In [9]:
indexes_to_keep = (end_of_sequence.index).tolist() + (end_of_sequence.index + 1).tolist()
indexes_to_keep.sort()

df.loc[indexes_to_keep]

,Time_utc,VehicleName,VehicleType,Pressure_Pa_1,Pressure_Pa_2,Pressure_Pa_3,Pressure_Pa_4,Pressure_Pa_5,Pressure_Pa_6,Temperature_K_1,...,epsilonX,epsilonY,epsilonZ,epsilonSpeed,AtmosphericPressure_Pa,dt_s,dd_m,RatioPayload,phaseNb,missing_record
9116,2023-01-01 13:17:57.353,C-132,Dumper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.061000,9.389000,41.400002,0.000000,59241.098324,5.003000e+00,62.058516,0.0,28.0,True
9117,2023-01-01 13:28:49.923,C-132,Dumper,9.790000e+05,986335.899998,842000.000000,786000.000000,806000.000000,848277.350000,340.15,...,11.776000,11.207000,41.400002,0.000000,61976.493133,6.525700e+02,NaN,0.0,28.0,True
34420,2023-01-03 01:23:16.486,C-132,Dumper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.355000,7.984000,32.200001,0.000000,NaN,5.077000e+00,36.072924,1.0,65.0,True
34421,2023-01-03 01:39:03.266,C-132,Dumper,9.030544e+05,911945.566669,787136.083328,NaN,765000.000000,800054.433331,316.15,...,8.267000,7.970000,27.600000,0.000000,59255.664828,9.467800e+02,NaN,1.0,65.0,False
38347,2023-01-03 11:17:37.285,C-132,Dumper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.155000,9.194000,23.000000,0.000000,NaN,4.961000e+00,49.997245,0.0,76.0,True
38348,2023-01-03 11:44:07.352,C-132,Dumper,9.626127e+05,963000.000000,824367.600000,756980.266667,795367.600000,832245.066667,333.15,...,15.040000,23.825001,55.200001,0.000000,62996.000000,1.590067e+03,NaN,0.0,76.0,False
59565,2023-01-04 17:52:55.543,C-132,Dumper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.723000,6.089000,18.400000,0.000000,58772.311781,4.935000e+00,0.000000,0.0,112.0,True
59566,2023-01-04 23:47:17.033,C-132,Dumper,8.560000e+05,NaN,751000.000000,683000.000000,718000.000000,761716.116668,289.15,...,9.703000,9.361000,27.600000,0.000000,58707.533468,1.672876e+09,NaN,NaN,NaN,False
59999,2023-01-05 01:17:04.185,C-132,Dumper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.522000,7.644000,27.600000,0.000000,NaN,2.484965e+03,NaN,0.0,114.0,True
60000,2023-01-05 01:51:05.000,C-132,Dumper,9.042500e+05,872250.000000,782916.666667,710833.333333,751833.333333,793833.333333,303.15,...,18.370001,12.357000,64.400002,36.740002,61249.131578,2.040815e+03,NaN,0.0,114.0,True


In [10]:
df2[df2["missing_value_tire"]]

NameError: name 'df2' is not defined

In [ ]:
df2["missing_value_tire"] & (~df2["missing_record"])

0         False
1         False
2         False
3         False
4         False
          ...  
201307    False
201308    False
201309    False
201310    False
201311    False
Length: 201312, dtype: bool